In [0]:
from pyspark.sql.functions import (
    col,
    sum,
    count,
    countDistinct,
    avg,
    min,
    max,
    when,
    round
)

In [0]:
orders_silver = spark.table("workspace.default.orders_silver")
customers_silver = spark.table("workspace.default.customers_silver")
products_silver = spark.table("workspace.default.products_silver")
sellers_silver = spark.table("workspace.default.sellers_silver")
order_items_silver = spark.table("workspace.default.order_items_silver")
payments_silver = spark.table("workspace.default.payments_silver")
reviews_silver = spark.table("workspace.default.reviews_silver")
category_translation_silver = spark.table(
    "workspace.default.category_translation_silver"
)

In [0]:
gold_order_items = (
    order_items_silver
    .join(
        orders_silver.select(
            "order_id",
            "customer_id",
            "order_status",
            "purchase_timestamp"
        ),
        on="order_id",
        how="left"
    )
    .join(
        products_silver.select(
            "product_id",
            "category_name"
        ),
        on="product_id",
        how="left"
    )
    .join(
        category_translation_silver,
        category_translation_silver["product_category_name"]
        == col("category_name"),
        how="left"
    )
    .join(
        sellers_silver.select(
            "seller_id",
            col("city").alias("seller_city"),
            col("state").alias("seller_state")
        ),
        on="seller_id",
        how="left"
    )
    .select(
        "order_id",
        "item_id",
        "customer_id",
        "product_id",
        "seller_id",
        "order_status",
        "purchase_timestamp",
        "category_name",
        "category_name_english",
        "seller_city",
        "seller_state",
        "shipping_limit_timestamp",
        "price",
        "freight_value"
    )
    .withColumn(
        "item_total",
        round(col("price") + col("freight_value"), 2)
    )
)

In [0]:
gold_order_items.show(10, truncate=False)

In [0]:
gold_order_items.printSchema()

In [0]:
print("Rows:", gold_order_items.count())

In [0]:
(
    gold_order_items.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_order_items")
)

In [0]:
spark.table(
    "workspace.default.gold_order_items"
).count()

In [0]:
gold_order_items.show(10, truncate=False)
print("Rows:", gold_order_items.count())

In [0]:
gold_customer_metrics = (
    gold_order_items
    .groupBy("customer_id")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        count("item_id").alias("total_items"),
        round(sum("price"), 2).alias("total_product_spend"),
        round(sum("freight_value"), 2).alias("total_freight_spend"),
        round(sum("item_total"), 2).alias("total_spend"),
        round(avg("item_total"), 2).alias("average_item_value"),
        min("purchase_timestamp").alias("first_purchase_timestamp"),
        max("purchase_timestamp").alias("last_purchase_timestamp")
    )
)


In [0]:
gold_customer_metrics.show(10, truncate=False)

In [0]:
print("Rows:", gold_customer_metrics.count())

In [0]:
(
    gold_customer_metrics.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_customer_metrics")
)

In [0]:
spark.table("workspace.default.gold_customer_metrics").count()

In [0]:
gold_product_metrics = (
    gold_order_items
    .groupBy(
        "product_id",
        "category_name",
        "category_name_english"
    )
    .agg(
        count("item_id").alias("total_items_sold"),
        countDistinct("order_id").alias("total_orders"),
        round(sum("price"), 2).alias("total_product_revenue"),
        round(sum("freight_value"), 2).alias("total_freight"),
        round(sum("item_total"), 2).alias("total_revenue"),
        round(avg("price"), 2).alias("average_price")
    )
)

In [0]:
gold_product_metrics.show(10, truncate=False)

In [0]:
print("Rows:", gold_product_metrics.count())

In [0]:
(
    gold_product_metrics.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_product_metrics")
)

In [0]:
spark.table("workspace.default.gold_product_metrics").count()

In [0]:
gold_sales_summary = (
    gold_order_items
    .agg(
        countDistinct("order_id").alias("total_orders"),
        count("item_id").alias("total_items_sold"),
        round(sum("price"), 2).alias("total_product_revenue"),
        round(sum("freight_value"), 2).alias("total_freight"),
        round(sum("item_total"), 2).alias("total_revenue"),
        round(
            sum("item_total") / countDistinct("order_id"),
            2
        ).alias("average_order_value")
    )
)

In [0]:
gold_sales_summary.show(truncate=False)

In [0]:
(
    gold_sales_summary.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_sales_summary")
)

In [0]:
spark.table("workspace.default.gold_sales_summary").show(truncate=False)

In [0]:
gold_delivery_metrics = (
    orders_silver
    .filter(col("delivered_customer_timestamp").isNotNull())
    .agg(
        count("order_id").alias("delivered_orders"),
        round(avg("delivery_days"), 2).alias("average_delivery_days"),
        min("delivery_days").alias("minimum_delivery_days"),
        max("delivery_days").alias("maximum_delivery_days"),
        round(avg("delivery_delay_days"), 2).alias("average_delivery_delay_days"),
        count(
            when(col("delivery_delay_days") > 0, True)
        ).alias("late_orders"),
        count(
            when(col("delivery_delay_days") <= 0, True)
        ).alias("on_time_orders")
    )
)

In [0]:
gold_delivery_metrics.show(truncate=False)

In [0]:
(
    gold_delivery_metrics.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_delivery_metrics")
)

In [0]:
spark.table("workspace.default.gold_delivery_metrics").show(truncate=False)

In [0]:
gold_order_items.explain()

In [0]:
gold_order_items.count()

In [0]:
gold_order_items.count()

In [0]:
import time

start_time = time.time()

row_count = gold_order_items.count()

end_time = time.time()

print(f"Rows: {row_count}")
print(f"Execution time: {end_time - start_time:.2f} seconds")